## Oracle Bulk Loader

This notebook serves as a conduit for bulk-loading multiple Oracle tables. It does so by executing the `oracle_to_bronze` procedure, compensating for the limited options available in Databricks Workflows.

#### Inputs:

1. **Subject Area**: This points to the directory containing all Oracle configurations that require execution.
2. **Push To Silver**: This flag determines whether data should also be migrated to the silver layer by executing the `bronze_to_silver` procedure.
3. **Parallel Executions**: This parameter dictates the number of concurrent executions. Note that increasing this value will also increase the number of simultaneous connections to Oracle.
4. **Notebook Timeout (seconds)**: This is the maximum allowed execution time for a notebook. If dealing with large Oracle tables, consider setting this parameter to a higher value.
5. **Configurations To Be Processed - View Only**: This shows the configurations to be processed. This parameter is not required when configuring a job to use oracle bulk loader.

In [0]:
import threading
import traceback
from concurrent.futures import ThreadPoolExecutor
from queue import Queue

import data.utilities.common as Commonlib
from beertech_utils.core.logger import LogManager
from pyspark.dbutils import DBUtils

In [0]:
# this entire command block will re-execute each time the "1. Subject Area" drop-down selection is changed - this populates the appropriate configurations in the "2. Configuration File" drop-down
CONFIG_BASE_PATH = "../../configuration/oracle/"
CONFIG_NOTEBOOK_PARALLEL_EXECUTIONS = ["1", "2", "4", "5", "10"]
CONFIG_NOTEBOOK_TIMEOUT = ["360", "3600", "36000"]
CONFIG_PUSH_TO_SILVER = ["yes", "no"]

# Configure the logging module
logger = LogManager.get_logger("Oracle Bulk Loader")

subject_entries = Commonlib.list_directories(CONFIG_BASE_PATH)
dbutils.widgets.dropdown("subject_area", "", [""] + sorted(subject_entries), "1. Subject Area")
subject_area = dbutils.widgets.get("subject_area")

config_selection_path = f"{CONFIG_BASE_PATH}{subject_area}"
all_config_files = Commonlib.list_files(config_selection_path, max_depth=1)
dbutils.widgets.dropdown(
    "config_file", "", [""] + sorted(all_config_files), "_*** Configurations To Be Processed - View Only ***"
)

# determine whether to push to silver
dbutils.widgets.dropdown("push_to_silver", "no", CONFIG_PUSH_TO_SILVER, "2. Push to Silver")
push_to_silver = dbutils.widgets.get("push_to_silver")

# parallel executions
dbutils.widgets.dropdown("parallel_executions", "1", CONFIG_NOTEBOOK_PARALLEL_EXECUTIONS, "3. Parallel Executions")
num_notebook_parallel_executions = dbutils.widgets.get("parallel_executions")

# notebook timeout
dbutils.widgets.combobox("notebook_timeout", "36000", CONFIG_NOTEBOOK_TIMEOUT, "4. Notebook Timeout (seconds)")
notebook_timeout = dbutils.widgets.get("notebook_timeout")

# notebook paths required
oracle_procedure = "oracle_to_bronze"
bronze_to_silver_procedure = "bronze_to_silver"

# parameters required
num_notebook_parallel_executions = int(num_notebook_parallel_executions)
notebook_timeout = int(notebook_timeout)

In [0]:

#### helper functions
def execute_notebook(notebook_path: str, parameters: dict) -> None:
    """
    Executes a Databricks notebook using the provided parameters.

    This function uses the DBUtils notebook.run function to execute a notebook, logging any exceptions that occur.

    Args:
        notebook_path (str): The path to the notebook that should be executed.
        parameters (dict): A dictionary of parameters to be passed to the notebook.

    Raises:
        Any exceptions raised during the execution of the notebook.
    """
    try:
        dbutils.notebook.run(notebook_path, timeout_seconds=notebook_timeout, arguments=parameters)
    except Exception as e:
        logger.error(f"failed to process notebook: notebook={notebook_path}, arguments={parameters}. error: {e}")
        # print the error traceback
        traceback.print_exc()


def process_tables() -> None:
    """
    Processes all tables from the execution queue.

    This function continuously pulls entries from the execution_queue and processes each one. An entry is processed by
    executing the corresponding "oracle_procedure" and "bronze_to_silver_procedure" notebooks for that table.
    The function logs the start and completion of the processing of each queue entry.

    This function will run indefinitely until the execution_queue is empty.

    Note:
        The execution_queue, dbutils and logger used in this function are assumed to be globally accessible.
    """
    while not execution_queue.empty():
        table = execution_queue.get()
        thread_id = threading.get_ident()
        logger.info(f"started processing table queue entry: thread_id={thread_id}, {table}")

        execute_notebook(table["oracle_procedure"], table["parameters"])

        if table.get("bronze_to_silver_procedure"):
            # add oracle to subject area, needed for bronze to silver
            table["parameters"]["subject_area"] = f"oracle/{table['parameters']['subject_area']}"
            execute_notebook(table["bronze_to_silver_procedure"], table["parameters"])

        execution_queue.task_done()
        logger.info(f"finished processing table queue entry: thread_id={thread_id}, {table}")


#### bulk loader main
logger.info(
    f"started processing oracle bulk loader. subject_area={subject_area}, push_to_silver={push_to_silver}, timeout={notebook_timeout}, parallel={num_notebook_parallel_executions}"
)

# create a DBUtils object
dbutils = DBUtils(spark)

# create a queue for table executions
execution_queue = Queue()

# fill the execution queue with tables to process
for single_config in all_config_files:
    execution_queue.put(
        {
            "oracle_procedure": oracle_procedure,
            "bronze_to_silver_procedure": bronze_to_silver_procedure if push_to_silver == "yes" else None,
            "parameters": {"subject_area": subject_area, "config_file": single_config},
        }
    )
    logger.info(f"filling queue with table: {execution_queue.queue[-1]}")

# create a ThreadPoolExecutor with the desired number of parallel executions
executor = ThreadPoolExecutor(max_workers=num_notebook_parallel_executions)

# start processing tables from the execution queue
for _ in range(num_notebook_parallel_executions):
    executor.submit(process_tables)

# wait for all notebooks to complete
execution_queue.join()

# shutdown the executor
executor.shutdown()

logger.info(
    f"ended processing oracle bulk loader. subject_area={subject_area}, push_to_silver={push_to_silver}, timeout={notebook_timeout}, parallel={num_notebook_parallel_executions}"
)